# 01 — Data Collection

Pulls NBA team game logs via `nba_api` for seasons 2020-21 through 2024-25.  
All raw data is saved to `data/raw/` immediately after fetching. Re-running this notebook skips files that already exist.

In [ ]:
import sys
from pathlib import Path


def _find_project_root() -> Path:
    """Locate the project root (the dir containing data/raw) regardless of
    where the Jupyter kernel was started from."""
    # 1. Walk up from the current working directory.
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    # 2. Fallback to the known absolute location of this project.
    fallback = Path("/Users/yourrem/PROJECTS/DATA_SCIENCE/Time_Series/sports_analytics")
    if (fallback / "data" / "raw").is_dir():
        return fallback
    raise FileNotFoundError(
        "Could not locate project root containing data/raw. "
        f"Searched from cwd={Path.cwd()}"
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.scraper import fetch_all_teams, fetch_team_game_logs

RAW_DIR = PROJECT_ROOT / "data" / "raw"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

SEASONS = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

## 1. All NBA Teams

In [2]:
teams_df = fetch_all_teams()
print(f"{len(teams_df)} teams loaded")
teams_df.head()

30 teams loaded


,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966


## 2. Team Game Logs

One CSV per team × season → `data/raw/team_game_logs_{team_id}_{season}.csv`

In [3]:
total = len(teams_df) * len(SEASONS)
done = 0

for _, team in teams_df.iterrows():
    for season in SEASONS:
        df = fetch_team_game_logs(team["id"], season)
        done += 1
        if done % 10 == 0 or done == total:
            print(f"  [{done}/{total}] {team['abbreviation']} {season} — {len(df)} games")

print("Done.")

  [10/150] BOS 2024-25 — 82 games
  [20/150] NOP 2024-25 — 82 games
  [30/150] DAL 2024-25 — 82 games
  [40/150] GSW 2024-25 — 82 games
  [50/150] LAC 2024-25 — 82 games
  [60/150] MIA 2024-25 — 82 games
  [70/150] MIN 2024-25 — 82 games
  [80/150] NYK 2024-25 — 82 games
  [90/150] IND 2024-25 — 82 games
  [100/150] PHX 2024-25 — 82 games
  [110/150] SAC 2024-25 — 82 games
  [120/150] OKC 2024-25 — 82 games
  [130/150] UTA 2024-25 — 82 games
  [140/150] WAS 2024-25 — 82 games
  [150/150] CHA 2024-25 — 82 games
Done.


## 3. Summary

In [ ]:
n_gamelogs = len(list(RAW_DIR.glob("team_game_logs_*.csv")))

print(f"Team game log files : {n_gamelogs}  (expect {len(teams_df) * len(SEASONS)} = 30×5)")

# quick row-count check on one team
sample = list(RAW_DIR.glob("team_game_logs_*.csv"))[0]
sample_df = pd.read_csv(sample)
print(f"\nSample file: {sample.name}")
print(f"  Rows: {len(sample_df)}, Columns: {list(sample_df.columns)}")